# MDC_51006_EP - Foundations of Machine Learning - Lab 5

The goals of this lab are:
- to play with tabular deep learning
- to showcase that recent advances can be easily available

We will rely on two really interesting articles:
- David Holzmüller, Léo Grinsztajn, et Ingo Steinwart. 2025. « Better by Default: Strong Pre-Tuned MLPs and Boosted Trees on Tabular Data ». arXiv:2407.04491. NeurIPS 2024. https://doi.org/10.48550/arXiv.2407.04491.
- Jingang Qu, David Holzmüller, Gaël Varoquaux, et Marine Le Morvan. 2025. « TabICL: A Tabular Foundation Model for In-Context Learning on Large Data ». arXiv:2502.05564. NeurIPS 2025. https://doi.org/10.48550/arXiv.2502.05564.




We will reuse what we have done previously. We are going to use the same dataset as well as the final metric.

In [ ]:
!uv pip install -U seaborn polars scikit-learn==1.6.1 numpy==2.1.3

Using Python 3.12.12 environment at: /usr
Resolved 21 packages in 125ms
Audited 21 packages in 1ms


In [ ]:
!wget -nc https://raw.githubusercontent.com/IBM/monitor-wml-model-with-watson-openscale/master/data/german_credit_data_biased_training.csv

File ‘german_credit_data_biased_training.csv’ already there; not retrieving.



In [ ]:
import polars as pl

In [ ]:
german_credit = pl.read_csv("german_credit_data_biased_training.csv")

In [ ]:
import seaborn.objects as so

In [ ]:
import polars.selectors as cs
string_columns = german_credit.select(cs.string()).columns
numerical_columns = german_credit.select(cs.numeric()).columns

In [ ]:
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
import numpy as np
categorical_features = string_columns.copy()
categorical_features.remove('Risk')
categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
numerical_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())
    ])

preprocessor = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer, categorical_features),
            ('num', numerical_transformer, numerical_columns)
        ],
        remainder='passthrough'
    )

from sklearn.preprocessing import OrdinalEncoder, FunctionTransformer
def convert_to_category(x):
   return(x.astype('category'))

categorical_transformer_cat = Pipeline(
    steps=[
        ("encoder", OrdinalEncoder(dtype=np.int32)),
        #("converter", FunctionTransformer(convert_to_category, check_inverse=False)),
    ]
)

preprocessor_cat = ColumnTransformer(
        transformers=[
            ('cat', categorical_transformer_cat, categorical_features),
            ('num', numerical_transformer, numerical_columns)
        ],
        remainder='passthrough'
    )

In [ ]:
real_prop = {'Risk': .02, 'No Risk': .98}
train_prop = {'Risk': 1/3, 'No Risk': 2/3}

In [ ]:
custom_weight = {'Risk': real_prop['Risk']/train_prop['Risk'], 'No Risk': real_prop['No Risk']/train_prop['No Risk']}

In [ ]:
def compute_costs(LoanAmount):
  return({'Risk_No Risk': 5.0 + .6 * LoanAmount, 'No Risk_No Risk': 1.0 - .05 * LoanAmount,
          'Risk_Risk': 1.0, 'No Risk_Risk': 1.0})

In [ ]:
def custom_metric(y_true, y_pred, LoanAmount):
  costs = compute_costs(LoanAmount)
  y_pred = pl.Series(y_pred)
  loss = (y_true=='Risk') * custom_weight['Risk'] *\
                ((y_pred=='Risk') * costs['Risk_Risk'] + (y_pred=='No Risk') * costs['Risk_No Risk']) +\
         (y_true=='No Risk') * custom_weight['No Risk'] *\
                ((y_pred=='Risk') * costs['No Risk_Risk'] + (y_pred=='No Risk') * costs['No Risk_No Risk'])
  return(loss.mean())

In [ ]:
sklearn.set_config(enable_metadata_routing=True)
custom_scorer = sklearn.metrics.make_scorer(custom_metric, greater_is_better=False).set_score_request(LoanAmount=True)

In [ ]:
import numpy as np
def custom_predict_from_proba(proba, LoanAmount):
  classes_ = np.array(['No Risk', 'Risk'])
  costs = compute_costs(LoanAmount)
  decision = np.vstack([( proba[:, 0] * custom_weight['No Risk'] * costs['No Risk_No Risk'] + proba[:, 1] * custom_weight['Risk'] * costs['Risk_No Risk']),
                       ( proba[:, 0] * custom_weight['No Risk'] * costs['No Risk_Risk'] + proba[:, 1] * custom_weight['Risk'] * costs['Risk_Risk'])])
  return(pl.Series(classes_[decision.argmin(axis=0)]))

def custom_predict_from_single_proba(proba, LoanAmount):
  proba=np.column_stack((1.0-proba, proba))
  return(custom_predict_from_proba(proba, LoanAmount))

def custom_predict(pipeline, X_test, LoanAmount):
  proba = pipeline.predict_proba(X_test)
  return(custom_predict_from_proba(proba, LoanAmount))

In [ ]:
def custom_metric_from_proba(y_true, proba, LoanAmount):
  y_pred = custom_predict_from_single_proba(proba, LoanAmount)
  return(custom_metric(y_true, y_pred, LoanAmount))

custom_scorer_from_proba = sklearn.metrics.make_scorer(custom_metric_from_proba, response_method="predict_proba", greater_is_better=False).set_score_request(LoanAmount=True, sample_weights=True)

In [ ]:
def compute_weights(y_true, LoanAmount):
  costs = compute_costs(LoanAmount)
  return(
      (y_true=='Risk') * custom_weight['Risk'] * (costs['Risk_No Risk']-costs['Risk_Risk'])
      + (y_true=='No Risk') * custom_weight['No Risk'] * (costs['No Risk_Risk']-costs['No Risk_No Risk'])
  )

In [ ]:
X = german_credit.drop('Risk')
y = german_credit['Risk']
LoanAmount = X['LoanAmount']

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test, LoanAmount_train, LoanAmount_test = train_test_split(X, y, LoanAmount, test_size=0.2, random_state=42)

In [ ]:
from sklearn.model_selection import cross_val_score

## Better by default

The algorithms described in the Better by Default article are available in the pytabkit package.

In [ ]:
!uv pip install pytabkit[models]

Using Python 3.12.12 environment at: /usr
Resolved 77 packages in 418ms
Prepared 7 packages in 1.22s
Installed 7 packages in 17ms
 + catboost==1.2.8
 + lightning-utilities==0.15.2
 + probmetrics==0.0.2
 + pytabkit==1.6.1
 + pytorch-lightning==2.5.5
 + skorch==1.2.0
 + torchmetrics==1.8.2


**Question:** Can you use use the `RealMLP_TD_Classifier` ?

**Answer:**

In [ ]:
from pytabkit import RealMLP_TD_Classifier

In [ ]:
pipeline_realmlp = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RealMLP_TD_Classifier())
    ])

In [ ]:
pipeline_realmlp.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/nn.py:794: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  new_y_cols.append(F.one_hot(y[[slice(None)] * (y.dim() - 1) + [i]], num_classes=cs).float())
INFO:pytorch_lightning.utilities.rank_zero:GPU availab

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['CheckingStatus',
                                                   'CreditHistory',
                                                   'LoanPurpose',
                                                   'ExistingSavings',
                                                   'EmploymentDuration', 'Sex',
                                                   'OthersOnLoan',
                                                   'OwnsProperty',
                                                   'InstallmentPlans',
                                                   'Housing', 'Job',
                                                   'Telephone',
                                                   'ForeignWorker']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['LoanDuration', 'LoanAmount',
                                                   'InstallmentPercent',
                                                   'CurrentResidenceDuration',
                                                   'Age',
                                                   'ExistingCreditsCount',
                                                   'Dependents'])])),
                ('classifier', RealMLP_TD_Classifier())])

In [ ]:
custom_scorer_from_proba(pipeline_realmlp, X_test, y_test, LoanAmount=LoanAmount_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


92.69603850000001

---

**Question:** Can you obtain a better result with hyperparameter optimization ?

**Answer:**

In [ ]:
from pytabkit import RealMLP_HPO_Classifier

In [ ]:
pipeline_realmlp_hpo = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RealMLP_HPO_Classifier())
    ])

In [ ]:
pipeline_realmlp_hpo.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/nn.py:794: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  new_y_cols.append(F.one_hot(y[[slice(None)] * (y.dim() - 1) + [i]], num_classes=cs).float())
INFO:pytorch_lightning.utilities.rank_zero:GPU availab

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['CheckingStatus',
                                                   'CreditHistory',
                                                   'LoanPurpose',
                                                   'ExistingSavings',
                                                   'EmploymentDuration', 'Sex',
                                                   'OthersOnLoan',
                                                   'OwnsProperty',
                                                   'InstallmentPlans',
                                                   'Housing', 'Job',
                                                   'Telephone',
                                                   'ForeignWorker']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['LoanDuration', 'LoanAmount',
                                                   'InstallmentPercent',
                                                   'CurrentResidenceDuration',
                                                   'Age',
                                                   'ExistingCreditsCount',
                                                   'Dependents'])])),
                ('classifier', RealMLP_HPO_Classifier())])

In [ ]:
custom_scorer_from_proba(pipeline_realmlp_hpo, X_test, y_test, LoanAmount=LoanAmount_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


94.15060650000001

---

**Question:** Are boosting or ensemble algorithms better (by default)?

**Anwser:**

In [ ]:
from pytabkit import LGBM_TD_Classifier

In [ ]:
pipeline_lgbm = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', LGBM_TD_Classifier())
    ])

In [ ]:
pipeline_lgbm.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['CheckingStatus',
                                                   'CreditHistory',
                                                   'LoanPurpose',
                                                   'ExistingSavings',
                                                   'EmploymentDuration', 'Sex',
                                                   'OthersOnLoan',
                                                   'OwnsProperty',
                                                   'InstallmentPlans',
                                                   'Housing', 'Job',
                                                   'Telephone',
                                                   'ForeignWorker']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['LoanDuration', 'LoanAmount',
                                                   'InstallmentPercent',
                                                   'CurrentResidenceDuration',
                                                   'Age',
                                                   'ExistingCreditsCount',
                                                   'Dependents'])])),
                ('classifier', LGBM_TD_Classifier())])

In [ ]:
custom_scorer_from_proba(pipeline_lgbm, X_test, y_test, LoanAmount=LoanAmount_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


96.62238299999999

In [ ]:
from pytabkit import Ensemble_TD_Classifier

In [ ]:
pipeline_ensemble = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', Ensemble_TD_Classifier())
    ])

In [ ]:
pipeline_ensemble.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


params["n_estimators"]=1000


/usr/local/lib/python3.12/dist-packages/pytabkit/models/nn_models/nn.py:794: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorch 2.9; use x[tuple(seq)] instead of x[seq]. In pytorch 2.9 this will be interpreted as tensor index, x[torch.tensor(seq)], which will result either in an error or a different result (Triggered internally at /pytorch/torch/csrc/autograd/python_variable_indexing.cpp:306.)
  new_y_cols.append(F.one_hot(y[[slice(None)] * (y.dim() - 1) + [i]], num_classes=cs).float())
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=256` reached.
INFO:pytorch_lightning.a

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['CheckingStatus',
                                                   'CreditHistory',
                                                   'LoanPurpose',
                                                   'ExistingSavings',
                                                   'EmploymentDuration', 'Sex',
                                                   'OthersOnLoan',
                                                   'OwnsProperty',
                                                   'InstallmentPlans',
                                                   'Housing', 'Job',
                                                   'Telephone',
                                                   'ForeignWorker']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['LoanDuration', 'LoanAmount',
                                                   'InstallmentPercent',
                                                   'CurrentResidenceDuration',
                                                   'Age',
                                                   'ExistingCreditsCount',
                                                   'Dependents'])])),
                ('classifier', Ensemble_TD_Classifier())])

In [ ]:
custom_scorer_from_proba(pipeline_ensemble, X_test, y_test, LoanAmount=LoanAmount_test)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


94.7948355

---

## TabICL

Implementation of tabICL are available in the `tabicl` package and in `autogluon`.

**Question:** Test those two implementations.

**Answer:**

In [ ]:
!uv pip install tabicl

Using Python 3.12.12 environment at: /usr
Resolved 60 packages in 391ms
Prepared 1 package in 24ms
Installed 1 package in 1ms
 + tabicl==0.1.3


In [ ]:
from tabicl import TabICLClassifier

In [ ]:
pipeline_tabicl = Pipeline(steps=[
        ('preprocessor', preprocessor_cat),
        ('classifier', TabICLClassifier())
    ])

In [ ]:
pipeline_tabicl.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO: You are downloading 'tabicl-classifier-v1.1-0506.ckpt', the latest best-performing version of TabICL.
To reproduce results from the original paper, please use 'tabicl-classifier-v1-0208.ckpt'.

Checkpoint 'tabicl-classifier-v1.1-0506.ckpt' not cached.



tabicl-classifier-v1.1-0506.ckpt:   0%|          | 0.00/108M [00:00<?, ?B/s]

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OrdinalEncoder(dtype=<class 'numpy.int32'>))]),
                                                  ['CheckingStatus',
                                                   'CreditHistory',
                                                   'LoanPurpose',
                                                   'ExistingSavings',
                                                   'EmploymentDuration', 'Sex',
                                                   'OthersOnLoan',
                                                   'OwnsProperty',
                                                   'InstallmentPlans',
                                                   'Housing', 'Job',
                                                   'Telephone',
                                                   'ForeignWorker']),
                                                 ('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  ['LoanDuration', 'LoanAmount',
                                                   'InstallmentPercent',
                                                   'CurrentResidenceDuration',
                                                   'Age',
                                                   'ExistingCreditsCount',
                                                   'Dependents'])])),
                ('classifier', TabICLClassifier())])

In [ ]:
custom_scorer_from_proba(pipeline_tabicl, X_test, y_test, LoanAmount=LoanAmount_test)

97.44232050000001

In [ ]:
!uv pip install autogluon


Using Python 3.12.12 environment at: /usr
Resolved 222 packages in 2.49s
Prepared 64 packages in 37.61s
Uninstalled 14 packages in 1.05s
Installed 64 packages in 641ms
 + adagio==0.2.6
 + aiohttp-cors==0.8.1
 + appdirs==1.4.4
 + autogluon==1.4.0
 + autogluon-common==1.4.0
 + autogluon-core==1.4.0
 + autogluon-features==1.4.0
 + autogluon-multimodal==1.4.0
 + autogluon-tabular==1.4.0
 + autogluon-timeseries==1.4.0
 + boto3==1.40.56
 + botocore==1.40.56
 + colorama==0.4.6
 + colorful==0.5.7
 + colorlog==6.10.1
 + coreforecast==0.0.16
 + distlib==0.4.0
 + einx==0.3.0
 + evaluate==0.4.6
 + fs==2.4.16
 + fugue==0.9.1
 + gluonts==0.16.2
 + jmespath==1.0.1
 - jsonschema==4.25.1
 + jsonschema==4.23.0
 + lightning==2.5.5
 - llvmlite==0.43.0
 + llvmlite==0.45.1
 + loguru==0.7.3
 + mlforecast==0.14.0
 + model-index==0.1.11
 + nlpaug==1.1.11
 - numba==0.60.0
 + numba==0.62.1
 - nvidia-cudnn-cu12==9.10.2.21
 + nvidia-cudnn-cu12==9.5.1.17
 - nvidia-cusparselt-cu12==0.7.1
 + nvidia-cusparselt-cu12==0

In [ ]:
from autogluon.tabular import TabularPredictor

In [ ]:
#concat X_train and y_train
train_data = pl.concat([X_train, y_train.to_frame()], how='horizontal').to_pandas()

In [ ]:
predictor = TabularPredictor(label='Risk').fit(train_data=train_data, hyperparameters={
        'TABICL': {}
    }, time_limit = 10*60)

No path specified. Models will be saved in: "AutogluonModels/ag-20251022_085706"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Oct  2 10:42:05 UTC 2025
CPU Count:          2
Memory Avail:       9.98 GB / 12.67 GB (78.8%)
Disk Space Avail:   68.37 GB / 112.64 GB (60.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU memory.
	presets='b

In [ ]:
custom_metric_from_proba(y_test, predictor.predict_proba(X_test.to_pandas())['Risk'], LoanAmount_test)

-96.88216650000001

---

**Question:** What about other tabular deep learning frameworks such as TABPFNV2 or Mitra ?

In [ ]:
!uv pip install autogluon.tabular[mitra]
!uv pip install autogluon.tabular[tabpfn]

Using Python 3.12.11 environment at: /usr
Audited 1 package in 112ms
Using Python 3.12.11 environment at: /usr
Resolved 71 packages in 261ms
Prepared 2 packages in 26ms
Installed 2 packages in 2ms
 + eval-type-backport==0.2.2
 + tabpfn==2.1.3


In [ ]:
predictor = TabularPredictor(label='Risk').fit(train_data=train_data, hyperparameters={
        'TABPFNV2': {
            # TabPFNv2 works best with default parameters on small datasets
        }
    }, time_limit = 10*60)

No path specified. Models will be saved in: "AutogluonModels/ag-20251012_111047"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sat Sep  6 09:54:41 UTC 2025
CPU Count:          2
Memory Avail:       9.81 GB / 12.67 GB (77.4%)
Disk Space Avail:   68.31 GB / 112.64 GB (60.7%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme' : New in v1.4: Massively better than 'best' on datasets <30000 samples by using new models meta-learned on https://tabarena.ai: TabPFNv2, TabICL, Mitra, and TabM. Absolute best accuracy. Requires a GPU. Recommended 64 GB CPU memory and 32+ GB GPU mem

tabpfn-v2-classifier-finetuned-zk73skhh.(…):   0%|          | 0.00/29.0M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/37.0 [00:00<?, ?B/s]

	0.822	 = Validation score   (accuracy)
	2.31s	 = Training   runtime
	3.76s	 = Validation runtime
Fitting model: WeightedEnsemble_L2 ... Training model for up to 360.00s of the 593.68s of remaining time.
	Ensemble Weights: {'TabPFNv2': 1.0}
	0.822	 = Validation score   (accuracy)
	0.0s	 = Training   runtime
	0.0s	 = Validation runtime
AutoGluon training complete, total runtime = 6.34s ... Best model: WeightedEnsemble_L2 | Estimated inference throughput: 132.8 rows/s (500 batch size)
Disabling decision threshold calibration for metric `accuracy` due to having fewer than 10000 rows of validation data for calibration, to avoid overfitting (500 rows).
	`accuracy` is generally not improved through threshold calibration. Force calibration via specifying `calibrate_decision_threshold=True`.
TabularPredictor saved. To load, use: predictor = TabularPredictor.load("/content/AutogluonModels/ag-20251012_111047")


In [ ]:
custom_metric_from_proba(y_test, predictor.predict_proba(X_test.to_pandas())['Risk'], LoanAmount_test)

-96.56416200000001

Mitra requires much more memory...

---

**Question** What is the performance of the default autogluon tabular predictor ?

**Answer:**

In [ ]:
predictor = TabularPredictor(label='Risk').fit(train_data=train_data, presets='high_quality', time_limit = 10*60)

No path specified. Models will be saved in: "AutogluonModels/ag-20251012_111059"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.12.11
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sat Sep  6 09:54:41 UTC 2025
CPU Count:          2
Memory Avail:       9.68 GB / 12.67 GB (76.4%)
Disk Space Avail:   68.22 GB / 112.64 GB (60.6%)
Presets specified: ['high_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
Note: `save_bag_folds=False`! This will greatly reduce peak disk usage during fit (by ~8x), but runs the risk of an out-of-memory error during model refit if memory is small relative to the data size.
	You can avoid th

In [ ]:
custom_metric_from_proba(y_test, predictor.predict_proba(X_test.to_pandas())['Risk'], LoanAmount_test)

-96.5158215